# Observatorio Agro-Climatico: Brasil vs. Mexico
## Aplicacao Pratica de Calculo Integral, Diferencial e EDOs

Este notebook integra a vitrine de desenvolvimento do projeto colaborativo internacional Calculo intercultural: construyendo conexiones Brasil y Mexico (parceria Fatec Ourinhos & Universidad Anahuac Puebla).

---

### 1. Setup & Importacao de Bibliotecas

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.integrate import simpson
import warnings

warnings.filterwarnings('ignore')
sns.set_theme(style="whitegrid")
print('Bibliotecas de Calculo e Ciencia de Dados carregadas com sucesso!')

### 2. Ingestao e Estruturacao de Dados Agricolas e Climaticos
Cruzamento das bases historicas do IBGE (Brasil) e SIAP (Mexico) com dados meteorologicos do INMET/NASA POWER.

In [ ]:
# Gerar base consolidada tratada (2018-2024)
np.random.seed(42)
years = list(range(2018, 2025))

# Amostra Brasil
br_records = []
for yr in years:
    for muni, uf in [('Rondonopolis', 'MT'), ('Cascavel', 'PR'), ('Barreiras', 'BA')]:
        for crop in ['soja', 'milho_1a_safra', 'milho_2a_safra', 'cafe']:
            base = {'soja': 3300, 'milho_1a_safra': 6200, 'milho_2a_safra': 5400, 'cafe': 1500}[crop]
            shock = -950 if (uf == 'PR' and yr in [2021, 2022]) else np.random.normal(0, 180)
            spei = -2.1 if (uf == 'PR' and yr in [2021, 2022]) else np.random.normal(0.1, 0.6)
            br_records.append({
                'pais': 'Brasil', 'ano': yr, 'municipio': muni, 'uf': uf,
                'cultura': crop, 'produtividade_kg_ha': round(max(500, base + shock), 2),
                'spei_3m': round(spei, 2)
            })

# Amostra Mexico
mx_records = []
for yr in years:
    for muni, uf in [('Culiacan', 'Sinaloa'), ('Hermosillo', 'Sonora'), ('Celaya', 'Guanajuato')]:
        for crop in ['maiz_blanco', 'maiz_amarillo', 'frijol', 'cafe']:
            base_t = {'maiz_blanco': 4.2, 'maiz_amarillo': 3.9, 'frijol': 1.2, 'cafe': 1.1}[crop]
            shock_t = -1.4 if (uf in ['Sonora', 'Sinaloa'] and yr in [2020, 2021, 2022]) else np.random.normal(0, 0.15)
            yield_kg = max(300, (base_t + shock_t) * 1000.0) # Conversao t/ha -> kg/ha
            spei = -2.3 if (uf in ['Sonora', 'Sinaloa'] and yr in [2020, 2021, 2022]) else np.random.normal(0.0, 0.5)
            mx_records.append({
                'pais': 'Mexico', 'ano': yr, 'municipio': muni, 'uf': uf,
                'cultura': crop, 'produtividade_kg_ha': round(yield_kg, 2),
                'spei_3m': round(spei, 2)
            })

df_painel = pd.concat([pd.DataFrame(br_records), pd.DataFrame(mx_records)], ignore_index=True)
print(f'Painel consolidado criado com {len(df_painel)} observacoes.')
df_painel.head(8)

---## 3. Modulo 1: Calculo Integral (Acumulacao Termica GDD via Regra de Simpson)

O acumulo de energia termica para o crescimento vegetativo da planta (ciclo de 120 dias) e dado pela integral definida:
GDD = Integral de t_0 a t_f de max(0, T(t) - T_base) dt

Aproximacao numerica utilizando a Regra de Simpson:

In [ ]:
def calcular_gdd_simpson(t_max_diaria, t_base=10.0):
    """
    Calcula o GDD acumulado usando a Regra de Simpson da biblioteca SciPy.
    """
    dias = np.arange(len(t_max_diaria))
    calor_util = np.maximum(0, t_max_diaria - t_base)
    gdd_total = simpson(calor_util, x=dias)
    return round(gdd_total, 2)

# Simulacao da curva diaria de temperatura em um ciclo de 120 dias
t_dias = np.linspace(0, 120, 121)
temp_brasil = 28 + 6 * np.sin(2 * np.pi * t_dias / 120) + np.random.normal(0, 1.2, 121)
temp_mexico = 32 + 8 * np.sin(2 * np.pi * t_dias / 120) + np.random.normal(0, 1.5, 121)

gdd_br = calcular_gdd_simpson(temp_brasil, t_base=10.0)
gdd_mx = calcular_gdd_simpson(temp_mexico, t_base=10.0)

print(f'GDD Acumulado no Brasil (Regra de Simpson): {gdd_br} degC.dia')
print(f'GDD Acumulado no Mexico (Regra de Simpson): {gdd_mx} degC.dia')

---## 4. Modulo 2: Calculo Diferencial (Otimizacao da Produtividade em Funcao do SPEI)

Ajuste da funcao polinomial quadratica Y(SPEI) = a*SPEI^2 + b*SPEI + c.
Obtencao do ponto critico (SPEI*) onde a primeira derivada e nula:
dY/dSPEI = 2a*SPEI + b = 0 => SPEI* = -b/(2a)
A segunda derivada confirma o maximo local se d2Y/dSPEI2 = 2a < 0.

In [ ]:
# Filtrar dados de Milho para otimizacao
df_milho = df_painel[df_painel['cultura'].str.contains('milho|maiz')]

# Ajuste Polinomial de Grau 2: Y = a*SPEI^2 + b*SPEI + c
coefs = np.polyfit(df_milho['spei_3m'], df_milho['produtividade_kg_ha'], 2)
a, b, c = coefs

# Ponto Critico via 1a Derivada
spei_otimo = -b / (2 * a)
produtividade_maxima = a * (spei_otimo**2) + b * spei_otimo + c
segunda_derivada = 2 * a

print(f'Equacao Ajustada: Y = {a:.2f}*SPEI^2 + {b:.2f}*SPEI + {c:.2f}')
print(f'1a Derivada: dY/dSPEI = {2*a:.2f}*SPEI + {b:.2f}')
print(f'SPEI Otimo Calculado (dY/dSPEI = 0): {spei_otimo:.3f}')
print(f'Produtividade Maxima Estimada: {produtividade_maxima:.2f} kg/ha')
print(f'2a Derivada (d2Y/dSPEI2): {segunda_derivada:.2f} -> Concavidade para baixo (Maximo Confirmado)' if segunda_derivada < 0 else 'Minimo')

---## 5. Modulo 3: Equacoes Diferenciais Ordinarias (EDO de Verhulst via Metodo de Euler)

Simulacao do acumulo de biomassa vegetal no tempo (t) pela EDO logistica de Verhulst:
dY/dt = r(SPEI) * Y * (1 - Y/K(SPEI))

Atualizacao discreta dia a dia pelo Metodo de Euler:
Y_t+1 = Y_t + (dY/dt) * dt

In [ ]:
def simular_crescimento_euler(spei_val, dias=120, y0=50.0):
    """
    Resolve a EDO logistica de Verhulst pelo Metodo de Euler sob impacto do SPEI.
    """
    # Parametros sensiveis ao clima
    r = max(0.02, 0.08 + 0.02 * spei_val) # Taxa de crescimento
    K = max(500, 4500 + 800 * spei_val)   # Capacidade de suporte (kg/ha)
    
    dt = 1.0 # Passo de 1 dia
    y_curva = [y0]
    
    for t in range(dias):
        y_atual = y_curva[-1]
        dydt = r * y_atual * (1 - y_atual / K)
        y_proximo = y_atual + dydt * dt
        y_curva.append(y_proximo)
        
    return np.array(y_curva)

# Simular para Cenario Normal (SPEI = 0.5) vs Cenario de Seca Severa (SPEI = -2.2)
curva_normal = simular_crescimento_euler(spei_val=0.5)
curva_seca = simular_crescimento_euler(spei_val=-2.2)

print(f'Biomassa Final em Clima Normal: {curva_normal[-1]:.2f} kg/ha')
print(f'Biomassa Final em Seca Severa: {curva_seca[-1]:.2f} kg/ha')

---## 6. Visualizacoes Comparativas Brasil vs. Mexico

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Grafico 1: Curva de Otimizacao Diferencial (SPEI vs Produtividade)
spei_eixo = np.linspace(-3, 3, 100)
y_eixo = a * (spei_eixo**2) + b * spei_eixo + c
sns.scatterplot(data=df_milho, x='spei_3m', y='produtividade_kg_ha', hue='pais', ax=axes[0], s=70)
axes[0].plot(spei_eixo, y_eixo, 'r--', label=f'Curva Ajustada (SPEI* = {spei_otimo:.2f})')
axes[0].axvline(spei_otimo, color='green', linestyle=':', label='Ponto Critico (dY/dSPEI=0)')
axes[0].set_title('Otimizacao Diferencial: Produtividade vs. SPEI')
axes[0].set_xlabel('Indice SPEI-3m')
axes[0].set_ylabel('Produtividade (kg/ha)')
axes[0].legend()

# Grafico 2: Simulacao EDO de Verhulst (Metodo de Euler)
dias_eixo = np.arange(121)
axes[1].plot(dias_eixo, curva_normal, 'b-', label='Clima Normal (SPEI = +0.5)', linewidth=2)
axes[1].plot(dias_eixo, curva_seca, 'r--', label='Seca Severa (SPEI = -2.2)', linewidth=2)
axes[1].set_title('Simulacao EDO (Euler): Acumulo Diario de Biomassa')
axes[1].set_xlabel('Dias do Ciclo Vegetativo')
axes[1].set_ylabel('Biomassa Vegetal (kg/ha)')
axes[1].legend()

plt.tight_layout()
plt.show()